# Assignment 8: Full Round 2-Style Problem — Flow Matching (100 points)

## Problem Description

In this problem, you will implement **Flow Matching**, a recent generative modeling framework that learns continuous normalizing flows via a simple regression objective. Flow matching provides a clean alternative to diffusion models with simpler training and sampling.

### Background: Continuous Normalizing Flows

A continuous normalizing flow (CNF) transforms a simple distribution $p_0$ (e.g., Gaussian noise) into a complex data distribution $p_1$ via an ODE:

$$\frac{dx}{dt} = v_\theta(x, t), \quad t \in [0, 1]$$

where $v_\theta$ is a learned velocity field parameterized by a neural network.

### Flow Matching Objective

Instead of training via maximum likelihood (expensive due to trace computation), flow matching uses a **conditional regression objective**:

Given:
- Source sample $x_0 \sim p_0 = \mathcal{N}(0, I)$ (noise)
- Target sample $x_1 \sim p_1$ (data)
- Interpolation path: $x_t = (1 - t) x_0 + t x_1$ (optimal transport path)
- Target velocity: $u_t(x_t | x_0, x_1) = x_1 - x_0$

The training loss is:

$$\mathcal{L}(\theta) = \mathbb{E}_{t \sim U[0,1], x_0 \sim p_0, x_1 \sim p_1} \left[ \|v_\theta(x_t, t) - (x_1 - x_0)\|^2 \right]$$

This is a simple MSE regression: the network learns to predict the velocity from noisy interpolated samples.

### Sampling

To generate samples, start from $x_0 \sim \mathcal{N}(0, I)$ and integrate the ODE:

$$x_{t+\Delta t} = x_t + \Delta t \cdot v_\theta(x_t, t)$$

using Euler integration from $t = 0$ to $t = 1$.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import math

---

> **WARNING:** Do not modify any code outside of the designated solution areas.

---

## Part 1: Interpolation Path Properties (8 points)

**[Non-coding]** The optimal transport interpolation path is $x_t = (1-t)x_0 + tx_1$.

1. (2 points) What is $x_t$ at $t = 0$? At $t = 1$? Verify that the path connects $x_0$ to $x_1$.

2. (2 points) Compute $\frac{dx_t}{dt}$. This is the target velocity $u_t$. Verify that it equals $x_1 - x_0$.

3. (2 points) Is the target velocity $u_t = x_1 - x_0$ constant along the path (independent of $t$)? What does this mean geometrically?

4. (2 points) Why is this called the "optimal transport" path? (Hint: what path minimizes the total distance traveled?)

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 2: Velocity Network (10 points)

**[Coding]** Implement the velocity network $v_\theta(x_t, t)$.

The network takes as input the concatenation of $x_t$ and $t$:
- $x_t \in \mathbb{R}^d$ — the noisy sample at time $t$
- $t \in \mathbb{R}$ — the time step

**Architecture:**
- Input: $[x_t; t] \in \mathbb{R}^{d+1}$
- Hidden: 3 layers of 256 units with SiLU (Swish) activation
- Output: $v \in \mathbb{R}^d$ (predicted velocity, same dimension as $x_t$)

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class VelocityNetwork(nn.Module):
    def __init__(self, d, hidden_dim=256, num_layers=3):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x_t, t):
        """
        Args:
            x_t: (B, d) noisy sample
            t: (B, 1) or (B,) time step
        Returns:
            v: (B, d) predicted velocity
        """
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 3: Training Step (12 points)

**[Coding]** Implement a single flow matching training step.

Given a batch of data $x_1 \sim p_1$:
1. Sample noise: $x_0 \sim \mathcal{N}(0, I)$
2. Sample time: $t \sim U[0, 1]$
3. Interpolate: $x_t = (1-t)x_0 + tx_1$
4. Target velocity: $u = x_1 - x_0$
5. Predict: $\hat{v} = v_\theta(x_t, t)$
6. Loss: $\|\hat{v} - u\|^2$

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def flow_matching_loss(model, x_1):
    """
    Compute the flow matching loss for a batch of data.
    
    Args:
        model: VelocityNetwork
        x_1: (B, d) batch of data samples
    
    Returns:
        loss: scalar
    """
    pass  # YOUR CODE

""" END OF THIS PART """

## Part 4: Euler ODE Solver (10 points)

**[Coding]** Implement Euler integration for sampling.

Starting from $x_0 \sim \mathcal{N}(0, I)$, integrate:
$$x_{t + \Delta t} = x_t + \Delta t \cdot v_\theta(x_t, t)$$

from $t = 0$ to $t = 1$ using $N$ steps (so $\Delta t = 1/N$).

In [ ]:
### WRITE YOUR SOLUTION HERE ###

@torch.no_grad()
def sample(model, num_samples, d, num_steps=100, device='cpu'):
    """
    Generate samples using Euler integration.
    
    Args:
        model: trained VelocityNetwork
        num_samples: number of samples to generate
        d: data dimension
        num_steps: number of Euler steps
    
    Returns:
        samples: (num_samples, d)
    """
    pass  # YOUR CODE

""" END OF THIS PART """

## Part 5: 2D Toy Dataset (6 points)

**[Coding]** Create a 2D dataset for visualization. Generate data from a mixture of 8 Gaussians arranged in a circle:

- Centers at $(3\cos(2\pi k/8), 3\sin(2\pi k/8))$ for $k = 0, \ldots, 7$
- Standard deviation $\sigma = 0.3$ per cluster
- 500 samples per cluster → 4000 total

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Generate 2D mixture of Gaussians

""" END OF THIS PART """

## Part 6: Training Loop (10 points)

**[Coding]** Train the flow matching model on the 2D dataset.

- 200 epochs, Adam optimizer, lr=1e-3
- Batch size 256
- Print loss every 50 epochs

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Training loop

""" END OF THIS PART """

## Part 7: Visualization (10 points)

**[Coding]** Visualize the results.

1. (4 points) Generate 1000 samples and plot them alongside the training data. Do the generated samples match the distribution?

2. (3 points) Visualize the learned velocity field: create a 2D grid of points, evaluate $v_\theta(x, t)$ at $t = 0.5$, and plot as a quiver (arrow) plot.

3. (3 points) Visualize the sampling trajectory: pick 10 starting points and show their paths from $t=0$ to $t=1$ (save intermediate positions during Euler integration).

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Visualization

""" END OF THIS PART """

## Part 8: Number of Integration Steps (8 points)

**[Coding]** Investigate how the number of Euler steps affects sample quality.

1. (5 points) Generate 500 samples each with $N = 1, 2, 5, 10, 50, 100$ Euler steps. For each, compute the mean distance to the nearest training data point (as a rough quality metric).

2. (3 points) Plot the quality metric vs. number of steps. How many steps are needed for good quality?

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Step count analysis

""" END OF THIS PART """

## Part 9: Alternative Interpolation Paths (10 points)

**[Non-coding + Coding]**

The optimal transport path $x_t = (1-t)x_0 + tx_1$ is not the only option. Another common choice is the **variance-preserving** (VP) path:

$$x_t = \cos\left(\frac{\pi t}{2}\right) x_0 + \sin\left(\frac{\pi t}{2}\right) x_1$$

1. (3 points) **[Non-coding]** Compute $\frac{dx_t}{dt}$ for the VP path. What is the target velocity?

2. (2 points) **[Non-coding]** Verify that $\|x_t\|^2 = \cos^2(\pi t/2)\|x_0\|^2 + \sin^2(\pi t/2)\|x_1\|^2 + 2\cos(\pi t/2)\sin(\pi t/2) x_0 \cdot x_1$. If $x_0$ and $x_1$ are independent with unit variance, what is $\mathbb{E}[\|x_t\|^2]$?

3. (5 points) **[Coding]** Train a flow matching model using the VP path. Compare generated samples with the OT path version.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# VP path flow matching

""" END OF THIS PART """

## Part 10: Time Conditioning Architecture (8 points)

**[Coding]** The simple concatenation of $t$ to the input is not the best way to condition on time. Implement a more sophisticated approach:

**Sinusoidal time embedding** (like in diffusion models):
$$\text{emb}(t) = [\sin(\omega_1 t), \cos(\omega_1 t), \sin(\omega_2 t), \cos(\omega_2 t), \ldots]$$
where $\omega_k = 10000^{-2k/d_{\text{emb}}}$.

Then project to hidden dimension and add to each hidden layer:
$$h_l = \text{SiLU}(W_l h_{l-1} + b_l) + \text{MLP}(\text{emb}(t))$$

1. (4 points) Implement the sinusoidal time embedding.
2. (4 points) Modify the velocity network to use this embedding. Retrain and compare with the simple concatenation approach.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, d_emb):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, t):
        # t: (B,) or (B, 1)
        # Returns: (B, d_emb)
        pass  # YOUR CODE


class ImprovedVelocityNetwork(nn.Module):
    def __init__(self, d, hidden_dim=256, d_emb=64, num_layers=3):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x_t, t):
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 11: Comparison with Diffusion (4 points)

**[Non-coding]**

1. (2 points) Flow matching trains with an MSE loss on the velocity. DDPM trains with an MSE loss on the noise. These are closely related. Show that predicting velocity $v = x_1 - x_0$ is equivalent to predicting noise $\epsilon = x_0$ when $x_t = (1-t)x_0 + tx_1$ (up to a scaling factor that depends on $t$).

2. (2 points) Flow matching uses deterministic ODE sampling. DDPM uses stochastic SDE sampling. What is the trade-off between the two approaches?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 12: Conditional Generation (8 points)

**[Coding]** Extend flow matching to class-conditional generation.

Modify the velocity network to also take a class label $y$ as input. Use a learnable class embedding.

1. (4 points) Implement `ConditionalVelocityNetwork` that takes $(x_t, t, y)$.
2. (4 points) Train on the 2D mixture data where each cluster has a label (0-7). Show that conditioning on a specific class generates samples from only that cluster.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class ConditionalVelocityNetwork(nn.Module):
    def __init__(self, d, num_classes, hidden_dim=256):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x_t, t, y):
        pass  # YOUR CODE

# Train and demonstrate conditional generation

""" END OF THIS PART """

## Part 13: Higher-Order ODE Solvers (4 points)

**[Coding]** Implement the **midpoint method** (2nd-order Runge-Kutta) for more accurate ODE integration:

$$k_1 = v_\theta(x_t, t)$$
$$k_2 = v_\theta(x_t + \frac{\Delta t}{2} k_1, t + \frac{\Delta t}{2})$$
$$x_{t+\Delta t} = x_t + \Delta t \cdot k_2$$

Compare sample quality with Euler at the same number of steps.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

@torch.no_grad()
def sample_midpoint(model, num_samples, d, num_steps=50, device='cpu'):
    pass  # YOUR CODE

""" END OF THIS PART """

## Part 14: Summary (2 points)

**[Non-coding]** In 3-4 sentences, summarize the key advantages of flow matching over DDPM-style diffusion models.

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """